# SQLite & CoinMarketCap API Notebook

This notebook provides a comprehensive workflow for analyzing historical and live Bitcoin (BTC-USD) market data. It combines data retrieval, storage, querying, and live updates using Python, SQLite, and public APIs. The notebook is organized for clarity, reproducibility, and ease of navigation, following best practices for data science notebooks

**Notebook Objectives:**
* Fetch and analyze 15 years of daily historical Bitcoin price and volume data.
* Store and manage data efficiently using SQLite databases.
* Retrieve and display live Bitcoin market data from CoinMarketCap.
* Update the database with the latest live data points.
* Provide clear code documentation and section-wise explanations for transparency and maintainability.


**Notebook Flow:**
1. Setup and Imports: Initialize the notebook environment and import required libraries.
2. API Configuration: Set up API keys, endpoints, and request parameters.
3. Historical Data Retrieval: Fetch and inspect long-term historical Bitcoin data.
4. Data Storage: Persist data in a local SQLite database.
5. Database Querying: Retrieve and analyze stored data.
6. Live Data Fetching: Obtain the latest Bitcoin market data from CoinMarketCap.
7. Database Update: Append new live data to the database.
8. Verification: Confirm successful updates and review the latest records.

**References, Citations Further Reading:**
API Documentation:
* sqlite.API: Details on the custom SQLite utilities used for data fetching, storage, and manipulation.
* SQLite3 Python Documentation: https://docs.python.org/3/library/sqlite3.html
* CoinMarketCap API Documentation: https://coinmarketcap.com/api/documentation/v1/
* Notebook structure and documentation style inspired by established data science best practices.

In [8]:
%load_ext autoreload
%autoreload 2
%autosave 30

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


Autosaving every 30 seconds


## Notebook Setup and Imports
This section sets up the notebook environment for efficient, interactive development and imports all necessary libraries for data fetching, manipulation, visualization, and storage.

In [23]:
# Import standard and third-party libraries for logging, HTTP requests, data handling, visualization, and database operations.
import logging
from requests import Request, Session
import pandas as pd
import sqlite3
import mplfinance as mpf
import yfinance as yf
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns
import sqlite_utils

# import helpers.hdbg as hdbg
# import helpers.hprint as hprint

## API Key and Endpoint Configuration
This section defines the API key and endpoint URLs for accessing live cryptocurrency data from CoinMarketCap. It also sets up request headers and parameters for API calls.

In [10]:
# hdbg.init_logger(verbosity=logging.INFO)

# _LOG = logging.getLogger(__name__)

# hprint.config_notebook()

In [11]:
# Set your CoinMarketCap API key.
API_KEY = #TO BE FILLED

# Define headers for API authentication and response format.
headers = {'X-CMC_PRO_API_KEY': API_KEY,'Accept': 'application/json'}

# Define the endpoint URL for fetching latest cryptocurrency listings.
url = 'https://pro-api.coinmarketcap.com/v1/cryptocurrency/listings/latest'

# Set request parameters: limit results and specify currency conversion.
params = {'limit': 1,'convert': 'USD'}

# Set the date format string for consistent datetime parsing.
DATEFMT = "%Y-%m-%d-%H-%M-%S"


# Testing API and Functions

## Load Data
This section retrieves 15 years of daily historical Bitcoin (BTC-USD) data using a custom utility, then displays the shape and a preview of the dataset.

In [24]:
# Fetch 15 years of daily BTC-USD historical data using a custom sqlite_utils function.
data = sqlite_utils.fetchHistoricalBTC(ticker="BTC-USD", period='15y', interval='1d')

# Print the shape (rows, columns) of the retrieved data.
print(f"Shape of data: {data.shape}")

# Display the first five rows to inspect the structure and sample values.
data.head()


Shape of data: (3869, 6)


,date,close,high,low,open,volume
0,2014-09-17-00-00-00,457.334015,468.174011,452.421997,465.864014,21056800
1,2014-09-18-00-00-00,424.440002,456.859985,413.104004,456.859985,34483200
2,2014-09-19-00-00-00,394.795990,427.834991,384.532013,424.102997,37919700
3,2014-09-20-00-00-00,408.903992,423.295990,389.882996,394.673004,36863600
4,2014-09-21-00-00-00,398.821014,412.425995,393.181000,408.084991,26580100


In [14]:
data.tail()

,date,close,high,low,open,volume
3864,2025-04-16-00-00-00,84033.867188,85428.281250,83100.617188,83674.507812,29617804112
3865,2025-04-17-00-00-00,84895.750000,85449.070312,83749.750000,84030.671875,21276866029
3866,2025-04-18-00-00-00,84450.804688,85095.046875,84298.882812,84900.187500,12728372364
3867,2025-04-19-00-00-00,85063.414062,85597.703125,84353.460938,84450.867188,15259300427
3868,2025-04-20-00-00-00,84586.343750,85291.500000,84012.218750,85071.359375,15549370368


## Storing Data into SQLite
This section stores the fetched historical Bitcoin data into a local SQLite database for persistent storage and future queries.

In [15]:
%%time
# Store the DataFrame into a local SQLite database file named 'btcDaily.db'.
sqlite_utils.storeData(data=data, db="./btcDaily.db")

CPU times: user 6.93 ms, sys: 3.25 ms, total: 10.2 ms
Wall time: 12.2 ms


## Querying Data from SQLite Database

This section retrieves the stored Bitcoin daily statistics from the SQLite database to verify storage and enable further analysis.

In [16]:
# Define SQL query to select all records from the btc_daily_stats table.
query = "SELECT * FROM btc_daily_stats"

# Fetch the data from the database.
data = sqlite_utils.fetchDB(query=query, db="./btcDaily.db")

In [17]:
print(f"Shape of data: {data.shape}")
data.head()

Shape of data: (3869, 6)


,date,close,high,low,open,volume
0,2014-09-17-00-00-00,457.334015,468.174011,452.421997,465.864014,21056800
1,2014-09-18-00-00-00,424.440002,456.859985,413.104004,456.859985,34483200
2,2014-09-19-00-00-00,394.795990,427.834991,384.532013,424.102997,37919700
3,2014-09-20-00-00-00,408.903992,423.295990,389.882996,394.673004,36863600
4,2014-09-21-00-00-00,398.821014,412.425995,393.181000,408.084991,26580100


## Fetching and Displaying Live Bitcoin Data

This section fetches the latest live Bitcoin data and auxiliary statistics from the CoinMarketCap API, then prints the results for immediate inspection.

In [26]:
# Define the endpoint for live Bitcoin quotes.
url = 'https://pro-api.coinmarketcap.com/v2/cryptocurrency/quotes/latest'

# Fetch live Bitcoin data and additional statistics using a custom utility.
btcLiveData, btcAuxData = sqlite_utils.liveBTC(url=url, API_KEY=API_KEY)

In [27]:
# Print the main live Bitcoin status (date, open price, volume).
print("Live Bitcoin Status:\n", btcLiveData)

Live Bitcoin Status:
 {'date': '2025-04-20-15-15-00', 'open': 84521.58302204874, 'volume': 14856297122.216606}


In [28]:
# Print additional details such as price changes, market cap, and dominance.
print("Live Bitcoin Additional Details:\n", btcAuxData)

Live Bitcoin Additional Details:
 {'price': 84521.58302204874, 'volume_24h': 14856297122.216606, 'volume_change_24h': 16.0852, 'percent_change_1h': -0.06432242, 'percent_change_24h': -1.05523572, 'percent_change_7d': 0.81723882, 'percent_change_30d': 0.89508851, 'percent_change_60d': -11.80297705, 'percent_change_90d': -21.24895747, 'market_cap': 1678084832114.697, 'market_cap_dominance': 62.9075, 'fully_diluted_market_cap': 1774953243463.02, 'tvl': None, 'last_updated': '2025-04-20T15:15:00.000Z'}


## Adding Live Data Point to Database
This section appends the latest live Bitcoin data point to the existing SQLite database, ensuring the dataset remains up to date.

In [21]:
# Add the latest live data point to the local SQLite database.
sqlite_utils.addDatapoint(liveData=btcLiveData, db="./btcDaily.db")

In [22]:
# Fetch and display the last few rows from the database to verify the new data point.
query = "SELECT * FROM btc_daily_stats"
data = sqlite_utils.fetchDB(query=query, db="./btcDaily.db")
data.tail()

,date,close,high,low,open,volume
3865,2025-04-17-00-00-00,84895.750000,85449.070312,83749.750000,84030.671875,2.127687e+10
3866,2025-04-18-00-00-00,84450.804688,85095.046875,84298.882812,84900.187500,1.272837e+10
3867,2025-04-19-00-00-00,85063.414062,85597.703125,84353.460938,84450.867188,1.525930e+10
3868,2025-04-20-00-00-00,84586.343750,85291.500000,84012.218750,85071.359375,1.554937e+10
3869,2025-04-20-15-06-00,NaN,NaN,NaN,84547.347507,1.555055e+10


Data point added to the last row